# Model 1 results: what drives a country's CO2 emissions?

Shows the results of `src/models/train_m1_emissions.py` in graphs and plain words.
Run the training script first; it saves everything to `reports/models/m1/` and logs the same files to the MLflow session run (Artifacts > comparison).

**The question:** when a country's population and income per person change, how much do its emissions change?

In [ ]:
import json, os
from pathlib import Path
import pandas as pd
from IPython.display import Image, display, Markdown

root = Path.cwd()
while not (root / "dvc.yaml").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
R = Path("reports/models/m1")
res = pd.read_csv(R / "comparison.csv")
summary = json.loads((R / "summary.json").read_text())
display(Markdown(f"**Champion:** `{summary['champion']}`  \n**Rule:** {summary['rule']}  \n**Decision:** {summary['reason']}"))

## The models, in plain words

### How the growth models work
Each growth model learns a rule like

> % change in emissions = a × % change in population + b × % change in income per person + drift × years

and predicts a future year by starting from the country's **last known emissions** and growing them by that rule.

**Worked example** (round made-up numbers): with a = 1.32, b = 0.56, drift = −0.4%/year, if a country's population grows 12% and income per person 55% over 10 years, emissions grow by about 1.32 × 11.3% + 0.56 × 43.8% − 0.4% × 10 ≈ 35.5% in log terms, i.e. **about 1.42 × the starting level**. (11.3% and 43.8% are the log versions of +12% and +55%.)

In [ ]:
desc = pd.Series(summary["descriptions"], name="what it is").to_frame()
desc.index.name = "model"
pd.set_option("display.max_colwidth", None)
desc

## How to read the scores

| Word | Plain meaning |
|---|---|
| **Stays the same** | The simplest guess: each country's emissions stay at their last known level. Every model must beat it. |
| **Skill vs stays the same** | How much smaller a model's error is than "stays the same". **+26% = about a quarter smaller.** Below 0 = worse than "stays the same". |
| **Exam** | Train only on years up to 1990, 2000 or 2010, then predict the next 12 years. Three exams show the model is not lucky once. |
| **Change R²** | The share of the real rise or fall in emissions the model explains. 0.40 = 40%. "Stays the same" always explains 0. |
| **a, b, drift** | The learned rule (see above). Drift below 0 = countries slowly getting cleaner per unit of income. |

In [ ]:
growth = res[res.model.str.startswith("growth")]
table = growth.pivot(index="model", columns="exam", values="skill_vs_stays_same_pct").round(1)
table.columns = [f"skill % (predict {c+1}-{c+12})" for c in table.columns]
table["average skill %"] = table.mean(axis=1).round(1)
table["change R² (2011-2022)"] = growth[growth.exam == 2010].set_index("model")["change_r2"].round(2)
table.sort_values("average skill %", ascending=False)

## Graph 1: skill in each exam

**How to read it:** each group of bars is one exam. The number on each bar is the skill. Above the black line = the model beat "stays the same".

**What to look for:** a reliable model stays clearly above 0 in **all three** exams.

In [ ]:
display(Image(R / "01_skill_by_exam.png"))

## Graph 2: skill by how far ahead

**How to read it:** one to three years ahead, emissions barely move, so "stays the same" is hard to beat. Further ahead, real changes build up, and population and income should help more.

**What to look for:** lines rising from left to right.

In [ ]:
display(Image(R / "02_skill_by_years_ahead.png"))

## Graph 3: how much of the real change each model explains

**How to read it:** the fairest score. "Stays the same" scores 0 here, because it never predicts any change. A bar that seems missing has a value near 0 (its number is printed). A bar below 0 means the model's predicted changes were worse than predicting no change at all.

In [ ]:
display(Image(R / "03_change_r2.png"))

## Graph 4: India, predicted from 2010

**How to read it:** black = India's real emissions. Grey dashed = "stays at the 2010 level". Coloured lines = each model's prediction for 2011-2022, made with data up to 2010 plus India's actual population and income changes.

**What to look for:** which coloured line follows the black line most closely.

In [ ]:
display(Image(R / "04_india_2011_2022.png"))

## Context: baselines and level models (exam 2011-2022 only)

`level_pop_income` predicts emissions directly from population and income. It explains differences *between* countries but ignores each country's own history, so it loses to "stays the same". `level_pop_income_energy` scores high mostly by the energy accounting rule.

In [ ]:
other = res[~res.model.str.startswith("growth")][["model", "r2", "typical_pct_error", "skill_vs_stays_same_pct"]]
other.round(3)

## Conclusions (generated from the numbers)

In [ ]:
champ = summary["champion"]
c = growth[growth.model == champ]
wins = int((c.skill_vs_stays_same_pct > 0).sum())
lines = [
    f"- **Champion:** `{champ}`.",
    f"- It beats 'stays the same' in **{wins} of 3** exams, with an average skill of **{c.skill_vs_stays_same_pct.mean():.1f}%**.",
    f"- In the 2011-2022 exam it explains **{c[c.exam == 2010].change_r2.iloc[0]:.0%}** of the real change in emissions; 'stays the same' explains none.",
    "- Decision details: " + summary["reason"],
]
display(Markdown("\n".join(lines)))